# ⚡ Módulo 14 - Notebook 01: PySpark Optimización ETL Pipelines

## 🚀 Performance y Pipelines de Producción

**Libro:** Saliendo de lo Pandito  
**Módulo:** 14 - PySpark Optimización ETL Pipelines  
**Duración estimada:** 80 minutos  
**Dificultad:** 🔴 Avanzado  
**Plataforma:** Databricks Free Edition

---

## 🎯 Objetivos de aprendizaje

Al finalizar este notebook serás capaz de:

✅ **Entender** el optimizador Catalyst  
✅ **Interpretar** planes de ejecución  
✅ **Optimizar** pipelines ETL  
✅ **Gestionar** particiones y caché  
✅ **Construir** ETL de producción

---

## 📋 Pre-requisitos

* ✅ Módulos 11-13 completados (PySpark Core, Transformaciones, SQL)
* ✅ Conocimiento de Spark DataFrames
* ✅ Familiaridad con conceptos ETL

---

## 📚 Contenido

1. Optimizador Catalyst
2. Planes de Ejecución (Logical/Physical)
3. Particionamiento (repartition/coalesce)
4. Caché y Persistencia
5. Data Skew y Soluciones
6. Caso Integrador: Pipeline ETL Completo

---

## 💡 Por qué importa

**Optimización = Ahorro de tiempo y dinero:**

* ⚡ **Performance:** 10-100x más rápido
* 💰 **Costo:** Menos recursos = menos gasto
* 📊 **Escala:** Procesar TB/PB de datos
* 🔧 **Producción:** Pipelines confiables

**La diferencia entre PoC y producción**

In [0]:
import pandas as pd
import numpy as np
from pyspark.sql import functions as F

print("💾 CARGANDO DATOS REALES DESDE UNITY CATALOG")
print("="*70)

CATALOG = "pandito_ds"
SCHEMA = "default"

try:
    # Cargar tabla de ventas de Los Andes Market (SPARK DataFrame)
    df = spark.table(f"{CATALOG}.{SCHEMA}.ventas_mensuales_mendoza_h3")
    
    print(f"\n✅ Datos reales cargados exitosamente")
    print(f"   📊 Registros: {df.count():,}")
    print(f"   🗂️ Particiones: {df.rdd.getNumPartitions()}")
    print(f"   📍 Ubicación: Mendoza, Argentina (Los Andes Market)")
    
    # Mostrar plan de ejecución
    print(f"\n📋 Plan de Ejecución Lógico:")
    df.explain(mode="simple")
    
    # Estadísticas del DataFrame
    print(f"\n📊 Estadísticas de particiones:")
    partitions_df = spark.sparkContext.parallelize(range(df.rdd.getNumPartitions()), df.rdd.getNumPartitions())
    sizes = df.rdd.glom().map(len).collect()
    print(f"   • Total particiones: {len(sizes)}")
    print(f"   • Registros por partición (min/max): {min(sizes)} / {max(sizes)}")
    print(f"   • Promedio por partición: {sum(sizes)/len(sizes):.0f}")
    
    # Detectar potencial data skew
    skew_ratio = max(sizes) / (sum(sizes) / len(sizes)) if sizes else 0
    if skew_ratio > 2:
        print(f"   ⚠️ SKEW DETECTADO: Ratio {skew_ratio:.2f}x (partición más grande vs promedio)")
    else:
        print(f"   ✅ Distribución balanceada (ratio: {skew_ratio:.2f}x)")
    
    print(f"\n🎯 Este notebook demostrará:")
    print(f"   • Catalyst Optimizer y sus optimizaciones")
    print(f"   • Cómo interpretar planes de ejecución")
    print(f"   • Técnicas de particionamiento")
    print(f"   • Cuándo usar cache/persist")
    
    USAR_DATOS_REALES = True
    
except Exception as e:
    print(f"\n⚠️  No se pudo cargar la tabla de Unity Catalog")
    print(f"   Error: {e}")
    print(f"\n📝 Solución:")
    print(f"   1. Ejecuta primero: 00_05_Preparacion_Datos_Empresariales.ipynb")
    print(f"   2. Verifica que la tabla exista: {CATALOG}.{SCHEMA}.ventas_mensuales_mendoza_h3")
    print(f"\n   Continuando con datos sintéticos...")
    
    df = None
    USAR_DATOS_REALES = False

print("\n" + "="*70)

## 📚 Optimización en Spark: El Motor Catalyst

### ⚙️ ¿Qué es Catalyst?

**Catalyst** es el optimizador de consultas de Spark.

**Proceso:**
```
1. Tu código (DataFrame API o SQL)
   ↓
2. Análisis (validación de esquema)
   ↓
3. Plan Lógico (qué hacer)
   ↓
4. Plan Físico (cómo hacerlo)
   ↓
5. Ejecución distribuida
```

**Ventaja:** Catalyst optimiza automáticamente tu código.

---

### 📋 Planes de Ejecución

**Tipos:**

**1️⃣ Plan Lógico:** Qué operaciones hacer
```python
df.filter("ventas > 100000").groupBy("zona").sum("ventas").explain(mode="simple")
```

**2️⃣ Plan Físico:** Cómo ejecutar (con optimizaciones)
```python
df.filter("ventas > 100000").groupBy("zona").sum("ventas").explain(mode="extended")
```

**Optimizaciones de Catalyst:**
* **Predicate Pushdown:** Filtrar antes de leer
* **Projection Pruning:** Leer solo columnas necesarias
* **Constant Folding:** Evaluar constantes una vez
* **Join Reordering:** Orden óptimo de joins

---

### 🗂️ Particionamiento

**Partición:** División de datos en bloques procesables.

**¿Por qué importa?**
* Cada partición = 1 tarea
* Muy pocas particiones = pobre paralelismo
* Demasiadas particiones = overhead

**Regla de oro:** 100-200 MB por partición

---

### 🔄 repartition() vs coalesce()

**repartition():** Redistribuye datos (full shuffle)
```python
# Aumentar o disminuir particiones
df_repart = df.repartition(100)  # Shuffle completo

# Por columna (para joins eficientes)
df_repart = df.repartition("zona")  # Datos de misma zona en misma partición
```

**coalesce():** Reducir particiones (sin shuffle)
```python
# Solo para REDUCIR particiones (más eficiente)
df_coalesced = df.coalesce(10)  # Sin shuffle
```

**Cuándo usar cada uno:**
* `repartition()`: Antes de joins/aggregations pesadas
* `coalesce()`: Antes de escribir (consolidar archivos)

---

### 💾 Cache y Persistencia

**Problema:** Re-computar un DataFrame costoso múltiples veces.

**Solución:** Almacenar en memoria.

**cache():** Almacena en memoria
```python
df_cached = df.filter("ventas > 100000").cache()

# Primera vez: computa y guarda en memoria
df_cached.count()  # Lento

# Subsecuentes: lee de memoria
df_cached.count()  # Rápido
```

**persist():** Control granular
```python
from pyspark import StorageLevel

# Solo memoria
df.persist(StorageLevel.MEMORY_ONLY)

# Memoria + disco (si no cabe)
df.persist(StorageLevel.MEMORY_AND_DISK)

# Serializado (menos RAM, más CPU)
df.persist(StorageLevel.MEMORY_ONLY_SER)
```

**Cuándo usar:**
* ✅ DataFrame usado múltiples veces
* ✅ Operaciones iterativas (ML)
* ❌ DataFrame usado solo una vez

**Limpiar caché:**
```python
df.unpersist()
```

---

### ⚠️ Data Skew

**Data Skew:** Una partición tiene MUCHO más datos que otras.

**Problema:**
```
Partición 1: 1,000 registros  ← Termina rápido
Partición 2: 1,200 registros  ← Termina rápido
Partición 3: 998,800 registros ← CUELLO DE BOTELLA (todo el job espera)
```

**Causas:**
* Claves con valores muy frecuentes (ej: "NULL", "UNKNOWN")
* Joins por claves desbalanceadas

**Solución: Salting**
```python
# Agregar sal aleatoria a la clave
df = df.withColumn(
    "clave_salted",
    F.concat(F.col("clave"), F.lit("_"), (F.rand() * 10).cast("int"))
)

# Ahora 1 clave → 10 claves (distribuye la carga)
```

---

### 🏗️ Pipeline ETL de Producción

**Estructura típica:**

```python
# 1. Extracción (Extract)
df_raw = spark.read.parquet("/raw/ventas")

# 2. Transformación (Transform)
df_clean = df_raw \
    .filter("ventas > 0") \
    .withColumn("año", F.year("fecha")) \
    .repartition("año")  # Optimizar para queries por año

# 3. Agregaciones
df_agg = df_clean \
    .groupBy("año", "zona") \
    .agg(
        F.sum("ventas").alias("ventas_totales"),
        F.count("*").alias("transacciones")
    ) \
    .cache()  # Re-usado múltiples veces

# 4. Carga (Load)
df_agg.write \
    .mode("overwrite") \
    .partitionBy("año") \
    .parquet("/processed/ventas_agregadas")
```

---

### 🎯 Best Practices

**1️⃣ Filtrar temprano**
```python
# ❌ MAL
df.groupBy("zona").sum("ventas").filter("ventas > 100000")

# ✅ BIEN
df.filter("ventas > 100000").groupBy("zona").sum("ventas")
```

**2️⃣ Proyectar solo columnas necesarias**
```python
# ❌ MAL
df.select("*").filter(...)

# ✅ BIEN
df.select("zona", "ventas").filter(...)
```

**3️⃣ Repartir antes de joins pesados**
```python
df1_repart = df1.repartition("clave")
df2_repart = df2.repartition("clave")
df_joined = df1_repart.join(df2_repart, "clave")
```

**4️⃣ Usar broadcast para tablas pequeñas**
```python
from pyspark.sql.functions import broadcast
df_large.join(broadcast(df_small), "clave")
```

**5️⃣ Coalesce antes de escribir**
```python
df.coalesce(10).write.parquet("/output")  # 10 archivos en lugar de 200
```

In [0]:
import pandas as pd
import numpy as np
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark import StorageLevel
from pyspark.sql.types import *
import warnings
warnings.filterwarnings('ignore')

print("⚡ PYSPARK OPTIMIZACIÓN ETL PIPELINES")
print("="*70)

print(f"\nVersión de Pandas: {pd.__version__}")
print(f"Versión de NumPy: {np.__version__}")

try:
    print(f"Versión de Spark: {spark.version}")
    
    # Configuración de Spark para optimización
    print(f"\n⚙️ Configuración actual de Spark:")
    print(f"   • Particiones default: {spark.conf.get('spark.sql.shuffle.partitions')}")
    print(f"   • Broadcast threshold: {spark.conf.get('spark.sql.autoBroadcastJoinThreshold')}")
except:
    print("⚠️  SparkSession no disponible")

print("\n🎯 En este notebook aprenderás:")
print("  • Catalyst Optimizer y planes de ejecución")
print("  • df.explain() - Interpretar planes")
print("  • df.repartition(n) / df.coalesce(n)")
print("  • df.cache() / df.persist(StorageLevel)")
print("  • Detectar y resolver data skew")

print("\n📖 Métodos clave:")
print("  - df.explain(mode='extended')  # Ver plan completo")
print("  - df.repartition(100, 'columna')  # Redistribuir")
print("  - df.coalesce(10)  # Reducir particiones sin shuffle")
print("  - df.cache() / df.persist()  # Almacenar en memoria")
print("  - df.rdd.getNumPartitions()  # Ver particiones")

print("\n" + "="*70)
print("✅ Librerías cargadas correctamente")

In [0]:
print("Módulo 14: Pipelines ETL de producción y optimización Catalyst en Spark")

